# Dropout strength in a ReLU MLP

A scheduling advantage at one dropout rate could be an accident of that rate. Here we vary the mean dropout probability at fixed depth and width on CIFAR-10, comparing uniform dropout with dropout concentrated in the first half or first third of the hidden layers. Rounding the number of active layers changes their individual rates, which we adjust to preserve the mean budget; the no-dropout run is shared across the sweep.

For the saved results, run **Setup**, then **Saved results** and **Figures**. Running **Training** repeats the sweep with three seeds per configuration and 125 epochs, as in the archived runs.


## Setup


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "utils").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "utils").is_dir():
    raise FileNotFoundError("Open this notebook from the repository root or notebooks/.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RESULTS_DIR = ROOT / "results"
RUN_DIR = ROOT / "runs" / "mlp_dropout_sweep"
RUN_DIR.mkdir(parents=True, exist_ok=True)
RESULT_FILE = "mlp_dropout_sweep.npz"

import torch
import numpy as np
import matplotlib.pyplot as plt
import ast
import time
from benchmarks.original.relu import get_dropout_schedule, CriticalReLUNet, train_epoch, evaluate
from functools import partial
from benchmarks.original import relu
from utils.cifar import load_cifar10 as load_cifar10_data
from utils.results import save_npz_result, load_npz_result
from scipy.stats import sem as standard_error

from utils.experiments import run_simulations as run_mlp_simulations
from utils.sweep_plots import (
    DISPLAY_NAMES,
    sweep_style,
    sweep_accuracy,
    sweep_advantage,
    xi_accuracy,
    sweep_table,
    training_panels,
    all_training_curves,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

if device == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    print(f"GPU: {torch.cuda.get_device_name(0)}")

USE_AMP = device == "cuda"
AMP_DTYPE = torch.bfloat16 if (USE_AMP and torch.cuda.is_bf16_supported()) else torch.float16

In [ ]:
# Configuration

N_SIMULATIONS = 3

# h_bar sweep values
H_BAR_VALUES = [0.005, 0.01, 0.025, 0.05, 0.1, 0.15]

# Schedules to compare
SCHEDULES = ["none", "constant", "reverse_step", "big_step"]

# Network geometry (fixed across all h_bar)
SIGMA_B_SQ = 0.00
SIGMA_W_SQ = 2
DEPTH_MULTIPLIER = 2
WIDTH = 256

# Depth is fixed using the h_bar = 0.1 baseline correlation length
# so architecture is identical across all h_bar values.
kappa = 2 * np.sqrt(2) / (3 * np.pi)
alpha = (3 / 2) * kappa ** (2 / 3)
H_BAR_REF = 0.1
xi_ref = 1 / (alpha * H_BAR_REF ** (1 / 3))
DEPTH = int(DEPTH_MULTIPLIER * xi_ref)

# Training
EPOCHS = 125
LEARNING_RATE = 1e-4
BATCH_SIZE = 75
USE_LR_SCHEDULE = True
LR_MIN = 1e-7

# Dataset
USE_FULL_DATASET = False
TRAIN_SIZE = None if USE_FULL_DATASET else 5000
TEST_SIZE = None if USE_FULL_DATASET else 5000

# Plotting — selected h_bar for Figure 4 (single-h_bar training curves)
H_BAR_PLOT = 0.1  # must be in H_BAR_VALUES

# Display / colour / marker maps (shared by all figures)
print(f"Fixed depth: L = {DEPTH}  (from {DEPTH_MULTIPLIER} x xi at h_bar = {H_BAR_REF})")
print(f"Width: {WIDTH}")
print(f"L/N = {DEPTH / WIDTH:.4f}")
print(f"h_bar values: {H_BAR_VALUES}")
print(f"Schedules: {SCHEDULES}")
print(f"Simulations per config: {N_SIMULATIONS}")
total_runs = len(H_BAR_VALUES) * (len(SCHEDULES) - 1) + 1  # 'none' only once
print(f"Total training runs: {total_runs * N_SIMULATIONS}")

## Correlation length

`H_BAR_VALUES` and `h_layers` store raw dropout probabilities $p_\ell$, despite their historical names. The calculation below substitutes them for the theoretical field $h_\ell=1-F_\ell(1)$ in the near-critical ReLU proxy, $\xi_{\mathrm{eff}}^{-1}\propto\langle p_\ell^{1/3}\rangle$. This gives a schedule-dependent scale to compare with training, without assuming that the finite network maintains a stationary variance.


In [ ]:
def compute_effective_xi(h_layers):
    """Compute effective correlation length for ReLU (kinked) networks."""
    L = len(h_layers)
    damage = sum(h ** (1 / 3) for h in h_layers if h > 0) / L
    xi_inv = alpha * damage
    return 1 / xi_inv if xi_inv > 0 else float("inf")

In [ ]:
# Theoretical predictions for every (h_bar, schedule) pair

theory = {}

print(f"{'h_bar':<8} {'Schedule':<16} {'h_mean':>8} {'h_max':>8} {'xi_eff':>8} {'L/xi':>6}")
print("-" * 60)

for hb in H_BAR_VALUES:
    h_max = 2 * hb
    for sched in SCHEDULES:
        h_layers = get_dropout_schedule(sched, DEPTH, hb, h_max)
        xi = compute_effective_xi(h_layers)
        theory[(hb, sched)] = {"h_layers": h_layers, "xi_eff": xi}

        xi_str = f"{xi:.2f}" if xi < 1e6 else "inf"
        ratio = DEPTH / xi if xi < 1e6 else 0
        print(
            f"{hb:<8.3f} {sched:<16} {np.mean(h_layers):>8.4f} "
            f"{max(h_layers):>8.4f} {xi_str:>8} {ratio:>6.2f}"
        )

## Training

This section downloads CIFAR-10 and trains the models with the configuration above. New results and figures are written to `runs/mlp_dropout_sweep/`.


In [ ]:
relu.USE_AMP, relu.AMP_DTYPE = USE_AMP, AMP_DTYPE
load_cifar10 = partial(load_cifar10_data, root=ROOT, device=device)
# Training utilities

In [ ]:
# Load data

print("Loading CIFAR-10...")
train_data, test_data = load_cifar10(TRAIN_SIZE, TEST_SIZE)
print(f"Train shape: {train_data[0].shape}")

In [ ]:
# Run the full h_bar sweep


def run_simulations(h_layers, n_sims, label):
    return run_mlp_simulations(
        lambda: CriticalReLUNet(3072, WIDTH, 10, h_layers, SIGMA_W_SQ, SIGMA_B_SQ).to(device),
        train_data,
        test_data,
        n_sims,
        label,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        lr_min=LR_MIN if USE_LR_SCHEDULE else None,
        train=train_epoch,
        evaluate=evaluate,
    )


all_results = {}
t_start = time.time()

# --- Run 'none' once (shared across all h_bar values) ---
print(f"\n{'=' * 60}")
print("Schedule: No dropout (run once, shared across all h_bar)")
print(f"{'=' * 60}")

none_result = run_simulations([0.0] * DEPTH, N_SIMULATIONS, "none")
best_none = none_result["test_acc"].max(axis=1)
print(f"No dropout: Best test = {best_none.mean():.2f}% +/- {standard_error(best_none):.2f}%")

for hb in H_BAR_VALUES:
    all_results[(hb, "none")] = none_result

# --- Run dropout schedules for each h_bar ---
for hb in H_BAR_VALUES:
    h_max = 2 * hb
    for sched in [s for s in SCHEDULES if s != "none"]:
        print(f"\n{'=' * 60}")
        print(f"h_bar = {hb:.3f} | Schedule: {sched}")
        print(f"{'=' * 60}")

        h_layers = get_dropout_schedule(sched, DEPTH, hb, h_max)
        xi = compute_effective_xi(h_layers)
        print(f"  h_layers = {[f'{h:.4f}' for h in h_layers]}")
        print(f"  xi_eff = {xi:.2f}, L/xi = {DEPTH / xi:.2f}")

        result = run_simulations(h_layers, N_SIMULATIONS, f"h={hb} {sched}")
        all_results[(hb, sched)] = result

        best = result["test_acc"].max(axis=1)
        print(f"  Best test = {best.mean():.2f}% +/- {standard_error(best):.2f}% (SEM)")

elapsed = time.time() - t_start
print(f"\nTotal time: {elapsed / 60:.1f} min")

In [ ]:
# Save results

save_data = {
    "all_results": {str(k): v for k, v in all_results.items()},
    "theory": {str(k): v for k, v in theory.items()},
    "config": {
        "H_BAR_VALUES": H_BAR_VALUES,
        "SCHEDULES": SCHEDULES,
        "N_SIMULATIONS": N_SIMULATIONS,
        "EPOCHS": EPOCHS,
        "DEPTH": DEPTH,
        "WIDTH": WIDTH,
        "SIGMA_W_SQ": SIGMA_W_SQ,
        "SIGMA_B_SQ": SIGMA_B_SQ,
        "LEARNING_RATE": LEARNING_RATE,
    },
}

save_npz_result(RUN_DIR / RESULT_FILE, save_data)

print(f"Results saved to {RUN_DIR / RESULT_FILE}")

## Saved results

The loader prefers a local rerun when one exists and otherwise reads the checked-in results. To compare against the archived runs regardless of what you have trained locally, set `result_path = RESULTS_DIR / RESULT_FILE`.


In [ ]:
result_path = RUN_DIR / RESULT_FILE
if not result_path.exists():
    result_path = RESULTS_DIR / RESULT_FILE
save_data = load_npz_result(result_path)

# Restore tuple keys: "(0.1, 'constant')" -> (0.1, 'constant')
all_results = {ast.literal_eval(k): v for k, v in save_data["all_results"].items()}
theory = {ast.literal_eval(k): v for k, v in save_data["theory"].items()}
config = save_data["config"]

# Restore the saved configuration
H_BAR_VALUES = config["H_BAR_VALUES"]
SCHEDULES = config["SCHEDULES"]
N_SIMULATIONS = config["N_SIMULATIONS"]
EPOCHS = config["EPOCHS"]
DEPTH = config["DEPTH"]
WIDTH = config["WIDTH"]
SIGMA_W_SQ = config["SIGMA_W_SQ"]
SIGMA_B_SQ = config["SIGMA_B_SQ"]
LEARNING_RATE = config["LEARNING_RATE"]

print(f"Loaded {len(all_results)} result entries from {result_path}")
print(f"h_bar values: {H_BAR_VALUES}")
print(f"Schedules: {SCHEDULES}")

In [ ]:
# Summary table

print(f"{'h_bar':<8} {'Schedule':<16} {'xi_eff':>8} {'L/xi':>6} {'Best Test Acc':>20}")
print("=" * 65)

for hb in H_BAR_VALUES:
    for sched in SCHEDULES:
        xi = theory[(hb, sched)]["xi_eff"]
        best = all_results[(hb, sched)]["test_acc"].max(axis=1)
        sem = standard_error(best)
        xi_str = f"{xi:.2f}" if xi < 1e6 else "inf"
        ratio = f"{DEPTH / xi:.2f}" if xi < 1e6 else "0.00"
        print(
            f"{hb:<8.3f} {DISPLAY_NAMES[sched]:<16} {xi_str:>8} {ratio:>6} "
            f"{best.mean():>8.2f} +/- {sem:.2f}%"
        )
    print("-" * 65)

## Figures

Accuracy summaries take each seed's highest recorded test accuracy, and the summary and paired-gain error bars show the sample standard error of the mean (SEM, `ddof=1`). The learning curves retain the original mean ± one standard deviation bands (`ddof=0`), which describe the spread across seeds; `scripts/plot_results.py` plots the same saved runs with SEM bands.


In [ ]:
sweep_style()
best = {key: result["test_acc"].max(axis=1) for key, result in all_results.items()}

In [ ]:
# Figure 1: Best test accuracy vs h_bar for each schedule

fig, ax = plt.subplots(figsize=(6, 4))

sweep_accuracy(ax, best, H_BAR_VALUES, SCHEDULES, capthick=1.2)

ax.set(xlabel=r"Mean dropout field $\bar{h}$", ylabel="Best test accuracy (%)")
ax.set_title(r"Dropout scheduling across $\bar{h}$ (MLP, CIFAR-10)")
ax.legend(loc="best")
ax.set_xlim(-0.005, max(H_BAR_VALUES) + 0.005)

plt.tight_layout()
plt.savefig(RUN_DIR / "h_bar_sweep_accuracy.pdf")
plt.show()
print(f"Saved: {RUN_DIR / 'h_bar_sweep_accuracy.pdf'}")

In [ ]:
# Figure 2: Scheduling advantage (step_early - constant) vs h_bar

fig, ax = plt.subplots(figsize=(6, 3.5))

sweep_advantage(ax, best, H_BAR_VALUES)

ax.set(xlabel=r"Mean dropout field $\bar{h}$", ylabel=r"$\Delta$ best test accuracy (%)")
ax.set_title(r"Scheduling advantage relative to constant dropout")
ax.legend(loc="best")

plt.tight_layout()
plt.savefig(RUN_DIR / "h_bar_sweep_delta.pdf")
plt.show()
print(f"Saved: {RUN_DIR / 'h_bar_sweep_delta.pdf'}")

In [ ]:
# Figure 3: Accuracy vs effective correlation length (all h_bar, all schedules)

fig, ax = plt.subplots(figsize=(6, 4))

xi_accuracy(ax, best, theory, H_BAR_VALUES)

ax.set(xlabel=r"Effective correlation length $\xi_{\mathrm{eff}}$", ylabel="Best test accuracy (%)")
ax.set_title(r"Accuracy vs $\xi_{\mathrm{eff}}$ across $\bar{h}$ values")
ax.legend(loc="best")

plt.tight_layout()
plt.savefig(RUN_DIR / "xi_vs_accuracy.pdf")
plt.show()
print(f"Saved: {RUN_DIR / 'xi_vs_accuracy.pdf'}")

In [ ]:
sweep_table(
    best,
    H_BAR_VALUES,
    SCHEDULES,
    caption=rf"Best test accuracy (\%) across dropout budgets $\bar{{h}}$ for different schedules (MLP, CIFAR-10). Errors are SEM over {N_SIMULATIONS} seeds.",
    label="tab:h_bar_sweep",
    heading=r"$\bar{h}$",
    value_format=".3f",
)

In [ ]:
fig = training_panels(
    all_results,
    H_BAR_PLOT,
    SCHEDULES,
    EPOCHS,
    title=rf"Training curves at $\bar{{h}} = {H_BAR_PLOT}$",
)
plt.savefig(RUN_DIR / f"h_bar_{H_BAR_PLOT}_training_curves.pdf", bbox_inches="tight")
plt.savefig(RUN_DIR / f"h_bar_{H_BAR_PLOT}_training_curves.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig = all_training_curves(
    all_results,
    H_BAR_VALUES,
    SCHEDULES,
    EPOCHS,
    row_labels=[rf"$\bar{{h}}={hb}$" for hb in H_BAR_VALUES],
)
plt.savefig(RUN_DIR / "h_bar_sweep_all_curves.pdf", bbox_inches="tight")
plt.savefig(RUN_DIR / "h_bar_sweep_all_curves.png", dpi=300, bbox_inches="tight")
plt.show()